# Multilingual Neuron Activation — Visualisation

Produces three publication-ready figures from the JSON activation results generated by the analysis notebook:

1. **Per-language, per-layer bar charts** — neuron counts across layers for each language, comparing all model variants  
2. **Aggregate bar chart with delta annotations** — total specialised neurons per language, with Δ labels showing shift from base  
3. **Cumulative layer curve** — how specialised neurons accumulate across transformer depth, with Kolmogorov–Smirnov distance vs. the base model  

Edit the **Configuration** section to point at your data and customise model labels, colours, and file names.

## Configuration

**Edit these variables before running the notebook.**

In [ ]:
# ── Google Drive ──────────────────────────────────────────────────────────────
DRIVE_BASE = "/content/drive/MyDrive/nlp_project_research"

# ── Model result files ────────────────────────────────────────────────────────
# Maps a display label to the JSON filename inside DRIVE_BASE.
# Add, remove, or rename entries here to change what is plotted.
# The first entry is treated as the "base" model for delta / KS calculations.
JSON_FILES = {
    "Qwen-2.5 Base": "QWEN_BASE.json",
    "Qwen-2.5 SFT":  "QWEN_SFT.json",
    "Qwen-2.5 GRPO": "QWEN_GRPO_CONSISTENT.json",
}

# ── Colours (one per model, in the same order as JSON_FILES) ─────────────────
# Defaults are colourblind-friendly. Must match the keys of JSON_FILES exactly.
MODEL_COLORS = {
    "Qwen-2.5 Base": "#bdc3c7",
    "Qwen-2.5 SFT":  "#2980b9",
    "Qwen-2.5 GRPO": "#c0392b",
}

# ── Layer tick positions shown on x-axes ─────────────────────────────────────
# Adjust if your model has a different number of layers.
LAYER_TICKS       = [1, 8, 16, 24, 32]
LAYER_TICK_LABELS = [1, 8, 16, 24, 32]

# ── Figure output (set to None to display inline only) ───────────────────────
SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"          # "pdf", "png", or "svg"
FIGURE_DPI    = 300

## Setup — Mount Drive and Import Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("Dependencies ready.")

## Plot Style and Helper Functions

In [ ]:
# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'serif',
    'axes.titlesize':    12,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        FIGURE_DPI,
    'grid.linestyle':    '--',
    'grid.alpha':        0.4,
    'axes.linewidth':    1.2,
})

# ── Derived constants ─────────────────────────────────────────────────────────
MODEL_ORDER = list(JSON_FILES.keys())
BASE_MODEL  = MODEL_ORDER[0]       # first entry is the reference / base


# ── I/O helpers ───────────────────────────────────────────────────────────────
def load_json(filename: str) -> dict | None:
    """Load a JSON file from DRIVE_BASE; return None and warn if missing."""
    path = os.path.join(DRIVE_BASE, filename)
    if not os.path.exists(path):
        print(f"  Warning: file not found — {filename}")
        return None
    with open(path) as f:
        return json.load(f)


def save_or_show(fig: plt.Figure, name: str) -> None:
    """Save figure to Drive (if SAVE_FIGURES) and display inline."""
    if SAVE_FIGURES:
        out_path = os.path.join(DRIVE_BASE, f"{name}.{FIGURE_FORMAT}")
        fig.savefig(out_path, bbox_inches='tight')
        print(f"  Saved: {out_path}")
    plt.show()


# ── Data-loading helpers ──────────────────────────────────────────────────────
def load_tidy(file_map: dict) -> pd.DataFrame | None:
    """
    Load all result JSONs and return a single tidy DataFrame with columns:
    [Layer, Language, Count, Model].
    """
    frames = []
    for label, filename in file_map.items():
        raw = load_json(filename)
        if raw is None:
            continue
        df = pd.DataFrame(raw).T
        df.index = df.index.astype(int)
        df = df.sort_index()
        melted = (
            df.reset_index()
              .melt(id_vars='index', var_name='Language', value_name='Count')
              .rename(columns={'index': 'Layer'})
        )
        melted['Model'] = label
        frames.append(melted)

    return pd.concat(frames, ignore_index=True) if frames else None


print(f"Base model: {BASE_MODEL}")
print(f"Model order: {MODEL_ORDER}")

## Load Data

In [ ]:
df = load_tidy(JSON_FILES)

if df is None:
    raise RuntimeError("No data loaded. Check that DRIVE_BASE and JSON_FILES are correct.")

languages = sorted(df['Language'].unique())
print(f"Loaded {len(df):,} rows | {len(languages)} languages | {df['Model'].nunique()} models")
print(f"Languages: {languages}")

## Figure 1: Per-Language, Per-Layer Neuron Counts

One subplot per language; x-axis = transformer layer, y-axis = number of specialised neurons, bars grouped by model variant.

In [ ]:
cols = 3
rows = math.ceil(len(languages) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 3.5 * rows), constrained_layout=True)
axes_flat = axes.flatten()

for i, lang in enumerate(languages):
    ax = axes_flat[i]
    subset = df[df['Language'] == lang]

    plot_data = (
        subset
        .pivot(index='Layer', columns='Model', values='Count')
        .reindex(columns=MODEL_ORDER)
    )
    plot_data.plot(
        kind='bar', ax=ax,
        color=[MODEL_COLORS[m] for m in MODEL_ORDER],
        width=0.85, edgecolor='black', linewidth=0.3,
    )

    ax.set_title(f"{lang.upper()}", fontweight='bold')
    ax.set_xlabel("Transformer Layer" if i >= len(languages) - cols else "")
    ax.set_ylabel("Specialised Neuron Count")
    ax.grid(axis='y', alpha=0.5)
    ax.legend(frameon=True, loc='upper left', fontsize=7)

    # Show only selected tick positions for readability
    tick_positions = [t - 1 for t in LAYER_TICKS]   # convert 1-indexed to bar positions
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(LAYER_TICK_LABELS, rotation=0)

# Hide any unused subplot panels
for j in range(len(languages), len(axes_flat)):
    axes_flat[j].axis('off')

save_or_show(fig, "fig1_per_language_per_layer")

## Figure 2: Aggregate Neuron Counts with Delta Annotations

Total specialised neurons summed across all layers, per language and model. Δ labels above the SFT and GRPO bars show the signed change relative to the base model.

In [ ]:
# Aggregate: sum counts across all layers
df_totals = (
    df.groupby(['Language', 'Model'])['Count']
      .sum()
      .reset_index()
)
plot_totals = (
    df_totals
    .pivot(index='Language', columns='Model', values='Count')
    .reindex(columns=MODEL_ORDER)
    .sort_index()
)

# Compute deltas relative to base model
for model in MODEL_ORDER[1:]:
    plot_totals[f'_delta_{model}'] = plot_totals[model] - plot_totals[BASE_MODEL]


def delta_labels(series: pd.Series) -> list[str]:
    """Format a series of delta values as annotated strings."""
    return [f"Δ {'+' if v > 0 else ''}{int(v)}" for v in series]


fig, ax = plt.subplots(figsize=(16, 8))

plot_totals[MODEL_ORDER].plot(
    kind='bar', ax=ax,
    color=[MODEL_COLORS[m] for m in MODEL_ORDER],
    width=0.8, edgecolor='black', linewidth=0.8,
)

# Annotate non-base bars with their delta vs base
for bar_idx, model in enumerate(MODEL_ORDER[1:], start=1):
    col = f'_delta_{model}'
    if col in plot_totals.columns:
        ax.bar_label(
            ax.containers[bar_idx],
            labels=delta_labels(plot_totals[col]),
            padding=6, fontsize=9, fontweight='bold',
            color=MODEL_COLORS[model], rotation=90,
        )

ax.set_title("Language-Specific Neuron Allocation — Delta Shifts from Base",
             fontweight='bold', fontsize=14, pad=20)
ax.set_xlabel("Language", fontweight='bold', labelpad=10)
ax.set_ylabel("Total Specialised Neuron Count", fontweight='bold', labelpad=10)
ax.set_xticklabels([l.upper() for l in plot_totals.index], rotation=0)
ax.set_ylim(0, plot_totals[MODEL_ORDER].max().max() * 1.55)
ax.grid(axis='y', alpha=0.4)

leg = ax.legend(frameon=True, loc='upper right',
                title=r"$\bf{Model\ Configuration}$",
                edgecolor='black')
leg.get_title().set_fontsize(10)

save_or_show(fig, "fig2_aggregate_delta")

## Figure 3: Cumulative Specialised Neurons Across Layers

Shows how neuron specialisation builds up as depth increases. The Kolmogorov–Smirnov statistic $D_{KS}$ quantifies how much each fine-tuned model's cumulative distribution diverges from the base.

In [ ]:
# Build cumulative sum per model across layers (summed over all languages)
df_layer_totals = (
    df.groupby(['Model', 'Layer'])['Count']
      .sum()
      .reset_index()
      .sort_values(['Model', 'Layer'])
)
df_layer_totals['CumulativeCount'] = (
    df_layer_totals.groupby('Model')['Count'].cumsum()
)

# Normalised CDFs for KS computation
normalised = {}
for model in MODEL_ORDER:
    sub = df_layer_totals[df_layer_totals['Model'] == model].sort_values('Layer')
    if sub.empty:
        continue
    values = sub['CumulativeCount'].values.astype(float)
    normalised[model] = values / values[-1]

base_cdf = normalised.get(BASE_MODEL)

def ks_label(model: str) -> str:
    if model == BASE_MODEL or base_cdf is None:
        return "Ref"
    if model not in normalised:
        return "N/A"
    d = np.max(np.abs(base_cdf - normalised[model]))
    return f"{d:.3f}"


# Marker styles — one per model
MARKERS = ['o', 's', '^', 'D', 'v', 'P']
LINES   = ['-', '--', '-', '-.', ':', '-']

fig, ax = plt.subplots(figsize=(10, 6))

for idx, model in enumerate(MODEL_ORDER):
    sub = df_layer_totals[df_layer_totals['Model'] == model].sort_values('Layer')
    if sub.empty:
        continue
    ax.plot(
        sub['Layer'], sub['CumulativeCount'],
        label=f"{model}  ($D_{{KS}}$: {ks_label(model)})",
        color=MODEL_COLORS[model],
        marker=MARKERS[idx % len(MARKERS)],
        markersize=5,
        linestyle=LINES[idx % len(LINES)],
        linewidth=2.2,
    )

ax.set_xlabel("Transformer Layer Index", fontweight='bold', labelpad=10)
ax.set_ylabel("Cumulative Specialised Neuron Count", fontweight='bold', labelpad=10)
ax.set_xticks(LAYER_TICKS)
ax.set_xticklabels(LAYER_TICK_LABELS)
ax.set_xlim(LAYER_TICKS[0], LAYER_TICKS[-1])
ax.grid(True, alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=False, loc='upper left')

save_or_show(fig, "fig3_cumulative_layers")